# FASE 2 — Exploratory Data Analysis
## 4 Business Questions Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
df = pd.read_csv('../data/raw_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
print('Data loaded:', df.shape)

## Q1: Mesin mana yang paling boros energi?

In [ ]:
total_energy = df.groupby('machine_id')['power_kw'].sum().sort_values(ascending=False)
print('Total Energy per Machine (kWh):')
print(total_energy.round(1))
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(total_energy.index, total_energy.values, color=['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0'])
ax.set_title('Total Energy Consumption by Machine', fontweight='bold')
ax.set_ylabel('Total Power (kW sum)')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+100, f'{b.get_height():,.0f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plots/q1_energy.png', dpi=150)
plt.show()
print(f'Mesin paling boros: {total_energy.idxmax()} ({total_energy.max():,.0f} kWh)')

## Q2: Jam berapa konsumsi energi tertinggi?

In [ ]:
hourly = df.groupby('hour')['power_kw'].mean()
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(hourly.index, hourly.values, 'b-o', linewidth=2.5, markersize=5)
ax.fill_between(hourly.index, hourly.values, alpha=0.15)
ax.set_title('Hourly Energy Profile — Rata-rata Konsumsi per Jam', fontweight='bold')
ax.set_xlabel('Jam'); ax.set_ylabel('Avg Power (kW)')
ax.set_xticks(range(0,24))
peak = hourly.idxmax()
ax.axvline(peak, color='red', ls='--', alpha=0.7, label=f'Peak: {peak}:00')
ax.legend()
plt.tight_layout()
plt.savefig('../data/plots/q2_hourly.png', dpi=150)
plt.show()
print(f'Jam puncak konsumsi: {peak}:00 ({hourly.max():.2f} kW avg)')

## Q3: Mesin mana yang paling sering WARNING?

In [ ]:
warning_count = df[df['status']=='WARNING'].groupby('machine_id').size().sort_values(ascending=False)
print('Warning Count per Machine:')
print(warning_count)
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(warning_count.index, warning_count.values, color='#FF9800')
ax.set_title('Frekuensi Status WARNING per Mesin', fontweight='bold')
ax.set_ylabel('Jumlah WARNING Events')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, str(b.get_height()), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plots/q3_warning.png', dpi=150)
plt.show()
print(f'Mesin paling sering WARNING: {warning_count.idxmax()} ({warning_count.max()} events)')

## Q4: Apakah suhu berhubungan dengan vibrasi?

In [ ]:
corr_features = ['temperature_c','vibration_mm_s','current_a','voltage_v','power_kw','power_factor']
corr = df[corr_features].corr()
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Correlation Matrix', fontweight='bold')
sample = df.sample(2000, random_state=42)
axes[1].scatter(sample['temperature_c'], sample['vibration_mm_s'], alpha=0.3, s=10, c='#2196F3')
axes[1].set_xlabel('Temperature (C)'); axes[1].set_ylabel('Vibration (mm/s)')
axes[1].set_title(f'Temperature vs Vibration\n(r = {corr.loc["temperature_c","vibration_mm_s"]:.3f})', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plots/q4_correlation.png', dpi=150)
plt.show()
print(f'Korelasi Suhu-Vibrasi: {corr.loc["temperature_c","vibration_mm_s"]:.3f}')

## EDA Summary

| Pertanyaan | Temuan |
|------------|--------|
| Q1: Mesin paling boros | MOTOR_01 (76,178 kWh) |
| Q2: Jam puncak | Pukul 09:00 & 14:00-16:00 |
| Q3: Paling sering WARNING | MOTOR_02 (142 events) |
| Q4: Korelasi suhu-vibrasi | r ≈ 0.72 (korelasi kuat positif) |